<a href="https://colab.research.google.com/github/ris2002/Multik30-Eng-De-_RNN_LSTM/blob/main/RNN_LSTM_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset
dataset = load_dataset("bentrevett/multi30k")
train_data=dataset['train']
test_data=dataset['test']
valid_data=dataset['validation']

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
len(train_data)

29000

In [ ]:
len(test_data)

1000

In [ ]:
len(valid_data)

1014

Data Checking

In [ ]:
for item in train_data:
  print(item['en'])
  print(item['de'])
  break



Two young, White males are outside near many bushes.
Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.




> Data Cleaning



In [ ]:
import re
def clean_data(text,lang):
  text=text.lower().strip()
  if lang=='en':
    text = re.sub(r"[^a-zA-Z\s]", "", text)
  elif lang=='de':
    text=re.sub(r"[^a-zA-ZäöüßÄÖÜ\s]", "", text)
  text = re.sub(r"\s+", " ", text) #cleans extra white space
  return text






In [ ]:
cleaned_results = []
for item in train_data:
  en_cleaned=clean_data(item['en'], 'en')
  de_cleaned=clean_data(item['de'],'de')
  cleaned_results.append({'en': en_cleaned, 'de': de_cleaned})


In [ ]:
for item in cleaned_results[:5]:
    print(f"EN: {item['en']}")
    print(f"DE: {item['de']}")
    print("-" * 10)

EN: two young white males are outside near many bushes
DE: zwei junge weiße männer sind im freien in der nähe vieler büsche
----------
EN: several men in hard hats are operating a giant pulley system
DE: mehrere männer mit schutzhelmen bedienen ein antriebsradsystem
----------
EN: a little girl climbing into a wooden playhouse
DE: ein kleines mädchen klettert in ein spielhaus aus holz
----------
EN: a man in a blue shirt is standing on a ladder cleaning a window
DE: ein mann in einem blauen hemd steht auf einer leiter und putzt ein fenster
----------
EN: two men are at the stove preparing food
DE: zwei männer stehen am herd und bereiten essen zu
----------


In [ ]:
cleaned_results[-1]


{'en': 'a man in shorts and a hawaiian shirt leans over the rail of a pilot boat with fog and mountains in the background',
 'de': 'ein mann in shorts und hawaiihemd lehnt sich über das geländer eines lotsenboots mit nebel und bergen im hintergrund'}

Tokenisation Pipeline


In [ ]:
!python -m spacy download de_core_news_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 34.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
from spacy.lang.en import English
from spacy.lang.de import German
nlp_en=English()
nlp_de=German()
def tokenize_eng(cleaned_results:list,en_tokens:list):

  tokenizer=nlp_en.tokenizer
  for text in cleaned_results:
    token_list=[]
    eng_text=text['en']

    doc=tokenizer(eng_text)
    for token in doc:
      token_list.append(token.text)
    en_tokens.append(token_list)

def tokenize_de(cleaned_results:list,de_tokens:list):

  tokenizer=nlp_de.tokenizer
  for text in cleaned_results:
    token_list=[]
    de_text=text['de']

    doc=tokenizer(de_text)
    for token in doc:
      token_list.append(token.text)
    de_tokens.append(token_list)







In [ ]:
en_tokens=[]
de_tokens=[]

In [ ]:
tokenize_eng(cleaned_results,en_tokens)
tokenize_de(cleaned_results,de_tokens)

In [ ]:
en_tokens[:5]

[['two',
  'young',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes'],
 ['several',
  'men',
  'in',
  'hard',
  'hats',
  'are',
  'operating',
  'a',
  'giant',
  'pulley',
  'system'],
 ['a', 'little', 'girl', 'climbing', 'into', 'a', 'wooden', 'playhouse'],
 ['a',
  'man',
  'in',
  'a',
  'blue',
  'shirt',
  'is',
  'standing',
  'on',
  'a',
  'ladder',
  'cleaning',
  'a',
  'window'],
 ['two', 'men', 'are', 'at', 'the', 'stove', 'preparing', 'food']]

In [ ]:
de_tokens[:5]

[['zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche'],
 ['mehrere',
  'männer',
  'mit',
  'schutzhelmen',
  'bedienen',
  'ein',
  'antriebsradsystem'],
 ['ein',
  'kleines',
  'mädchen',
  'klettert',
  'in',
  'ein',
  'spielhaus',
  'aus',
  'holz'],
 ['ein',
  'mann',
  'in',
  'einem',
  'blauen',
  'hemd',
  'steht',
  'auf',
  'einer',
  'leiter',
  'und',
  'putzt',
  'ein',
  'fenster'],
 ['zwei', 'männer', 'stehen', 'am', 'herd', 'und', 'bereiten', 'essen', 'zu']]

Build a vocab dictionary

In [ ]:
from collections import Counter
def build_dictionary(tokens,min_freq=2):
  counter=Counter()
  for text in tokens:
    counter.update(text)
  vocab={"<unk>": 0, "<pad>": 1, "<sos>": 2, "<eos>": 3}
  for word,freq in counter.items():
    if freq>=min_freq:
      vocab[word]=len(vocab)# mapping it to the current size of the 'vocab dict'
  itos={}#index_to_string
  for string,index in vocab.items():
    itos[index]=string
  return vocab,itos

In [ ]:
en_vocab,en_itos=build_dictionary(en_tokens)
de_vocab,de_itos=build_dictionary(de_tokens)

In [ ]:
en_vocab

{'<unk>': 0,
 '<pad>': 1,
 '<sos>': 2,
 '<eos>': 3,
 'two': 4,
 'young': 5,
 'white': 6,
 'males': 7,
 'are': 8,
 'outside': 9,
 'near': 10,
 'many': 11,
 'bushes': 12,
 'several': 13,
 'men': 14,
 'in': 15,
 'hard': 16,
 'hats': 17,
 'operating': 18,
 'a': 19,
 'giant': 20,
 'pulley': 21,
 'system': 22,
 'little': 23,
 'girl': 24,
 'climbing': 25,
 'into': 26,
 'wooden': 27,
 'playhouse': 28,
 'man': 29,
 'blue': 30,
 'shirt': 31,
 'is': 32,
 'standing': 33,
 'on': 34,
 'ladder': 35,
 'cleaning': 36,
 'window': 37,
 'at': 38,
 'the': 39,
 'stove': 40,
 'preparing': 41,
 'food': 42,
 'green': 43,
 'holds': 44,
 'guitar': 45,
 'while': 46,
 'other': 47,
 'observes': 48,
 'his': 49,
 'smiling': 50,
 'stuffed': 51,
 'lion': 52,
 'trendy': 53,
 'talking': 54,
 'her': 55,
 'cellphone': 56,
 'gliding': 57,
 'slowly': 58,
 'down': 59,
 'street': 60,
 'woman': 61,
 'with': 62,
 'large': 63,
 'purse': 64,
 'walking': 65,
 'by': 66,
 'gate': 67,
 'boys': 68,
 'dancing': 69,
 'poles': 70,
 'middl

In [ ]:
de_vocab

{'<unk>': 0,
 '<pad>': 1,
 '<sos>': 2,
 '<eos>': 3,
 'zwei': 4,
 'junge': 5,
 'weiße': 6,
 'männer': 7,
 'sind': 8,
 'im': 9,
 'freien': 10,
 'in': 11,
 'der': 12,
 'nähe': 13,
 'vieler': 14,
 'büsche': 15,
 'mehrere': 16,
 'mit': 17,
 'schutzhelmen': 18,
 'bedienen': 19,
 'ein': 20,
 'kleines': 21,
 'mädchen': 22,
 'klettert': 23,
 'spielhaus': 24,
 'aus': 25,
 'holz': 26,
 'mann': 27,
 'einem': 28,
 'blauen': 29,
 'hemd': 30,
 'steht': 31,
 'auf': 32,
 'einer': 33,
 'leiter': 34,
 'und': 35,
 'putzt': 36,
 'fenster': 37,
 'stehen': 38,
 'am': 39,
 'herd': 40,
 'bereiten': 41,
 'essen': 42,
 'zu': 43,
 'grün': 44,
 'hält': 45,
 'eine': 46,
 'gitarre': 47,
 'während': 48,
 'andere': 49,
 'sein': 50,
 'ansieht': 51,
 'lächelt': 52,
 'einen': 53,
 'ausgestopften': 54,
 'löwen': 55,
 'an': 56,
 'schickes': 57,
 'spricht': 58,
 'dem': 59,
 'handy': 60,
 'sie': 61,
 'langsam': 62,
 'die': 63,
 'straße': 64,
 'frau': 65,
 'großen': 66,
 'geldbörse': 67,
 'geht': 68,
 'tor': 69,
 'vorbei': 70

In [ ]:
en_itos

{0: '<unk>',
 1: '<pad>',
 2: '<sos>',
 3: '<eos>',
 4: 'two',
 5: 'young',
 6: 'white',
 7: 'males',
 8: 'are',
 9: 'outside',
 10: 'near',
 11: 'many',
 12: 'bushes',
 13: 'several',
 14: 'men',
 15: 'in',
 16: 'hard',
 17: 'hats',
 18: 'operating',
 19: 'a',
 20: 'giant',
 21: 'pulley',
 22: 'system',
 23: 'little',
 24: 'girl',
 25: 'climbing',
 26: 'into',
 27: 'wooden',
 28: 'playhouse',
 29: 'man',
 30: 'blue',
 31: 'shirt',
 32: 'is',
 33: 'standing',
 34: 'on',
 35: 'ladder',
 36: 'cleaning',
 37: 'window',
 38: 'at',
 39: 'the',
 40: 'stove',
 41: 'preparing',
 42: 'food',
 43: 'green',
 44: 'holds',
 45: 'guitar',
 46: 'while',
 47: 'other',
 48: 'observes',
 49: 'his',
 50: 'smiling',
 51: 'stuffed',
 52: 'lion',
 53: 'trendy',
 54: 'talking',
 55: 'her',
 56: 'cellphone',
 57: 'gliding',
 58: 'slowly',
 59: 'down',
 60: 'street',
 61: 'woman',
 62: 'with',
 63: 'large',
 64: 'purse',
 65: 'walking',
 66: 'by',
 67: 'gate',
 68: 'boys',
 69: 'dancing',
 70: 'poles',
 71: 'm

In [ ]:
de_itos

{0: '<unk>',
 1: '<pad>',
 2: '<sos>',
 3: '<eos>',
 4: 'zwei',
 5: 'junge',
 6: 'weiße',
 7: 'männer',
 8: 'sind',
 9: 'im',
 10: 'freien',
 11: 'in',
 12: 'der',
 13: 'nähe',
 14: 'vieler',
 15: 'büsche',
 16: 'mehrere',
 17: 'mit',
 18: 'schutzhelmen',
 19: 'bedienen',
 20: 'ein',
 21: 'kleines',
 22: 'mädchen',
 23: 'klettert',
 24: 'spielhaus',
 25: 'aus',
 26: 'holz',
 27: 'mann',
 28: 'einem',
 29: 'blauen',
 30: 'hemd',
 31: 'steht',
 32: 'auf',
 33: 'einer',
 34: 'leiter',
 35: 'und',
 36: 'putzt',
 37: 'fenster',
 38: 'stehen',
 39: 'am',
 40: 'herd',
 41: 'bereiten',
 42: 'essen',
 43: 'zu',
 44: 'grün',
 45: 'hält',
 46: 'eine',
 47: 'gitarre',
 48: 'während',
 49: 'andere',
 50: 'sein',
 51: 'ansieht',
 52: 'lächelt',
 53: 'einen',
 54: 'ausgestopften',
 55: 'löwen',
 56: 'an',
 57: 'schickes',
 58: 'spricht',
 59: 'dem',
 60: 'handy',
 61: 'sie',
 62: 'langsam',
 63: 'die',
 64: 'straße',
 65: 'frau',
 66: 'großen',
 67: 'geldbörse',
 68: 'geht',
 69: 'tor',
 70: 'vorbei'

In [ ]:
def mapping(vocab,text):
  mapped_lists=[]
  default="<unk>"
  for word in text:
    x=vocab.get(word,default)
    mapped_lists.append(x)
  return mapped_lists


In [ ]:
mapped_en_texts=[]
for text in en_tokens:
  en_mapped_lists=mapping(en_vocab,text)
  mapped_en_texts.append(en_mapped_lists)
mapped_de_texts=[]
for text in de_tokens:
  de_mapped_lists=mapping(de_vocab,text)
  mapped_de_texts.append(de_mapped_lists)



In [ ]:
len(mapped_en_texts)

29000

In [ ]:
len(mapped_de_texts)

29000

In [ ]:
mapped_en_texts

[[4, 5, 6, 7, 8, 9, 10, 11, 12],
 [13, 14, 15, 16, 17, 8, 18, 19, 20, 21, 22],
 [19, 23, 24, 25, 26, 19, 27, 28],
 [19, 29, 15, 19, 30, 31, 32, 33, 34, 19, 35, 36, 19, 37],
 [4, 14, 8, 38, 39, 40, 41, 42],
 [19, 29, 15, 43, 44, 19, 45, 46, 39, 47, 29, 48, 49, 31],
 [19, 29, 32, 50, 38, 19, 51, 52],
 [19, 53, 24, 54, 34, 55, 56, 46, 57, 58, 59, 39, 60],
 [19, 61, 62, 19, 63, 64, 32, 65, 66, 19, 67],
 [68, 69, 34, 70, 15, 39, 71, 72, 39, 73],
 [19, 74, 75, 72, 76, 77, 78, 15, 79],
 [80, 81, 82, 83, 17, 84, 85, 8, 78, 38, 39, 86, 72, 19, 87],
 [19, 88, 89, 90, 19, 91, 89, 8, 92],
 [19, 29, 15, 19, 93, 43, 90, 94, 95, 32, 96, 34, 19, 43, 97],
 [13, 98, 99, 9, 15, 19, 100],
 [19,
  101,
  15,
  19,
  88,
  86,
  62,
  102,
  32,
  103,
  104,
  '<unk>',
  34,
  19,
  '<unk>',
  105],
 [19, 23, 24, 32, 106, 15, 107, 72, 19, 63, 108, 109],
 [19, 29, 110, 34, 39, 111, 112, 113, 19, 6, 89, 32, 114, 115],
 [76, 116, 8, 106, 15, 19, 117, 62, 118],
 [19, 119, 72, 120, 98, 121, 122, '<unk>', 123, 1

In [ ]:
mapped_de_texts

[[4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
 [16, 7, 17, 18, 19, 20, '<unk>'],
 [20, 21, 22, 23, 11, 20, 24, 25, 26],
 [20, 27, 11, 28, 29, 30, 31, 32, 33, 34, 35, 36, 20, 37],
 [4, 7, 38, 39, 40, 35, 41, 42, 43],
 [20, 27, 11, 44, 45, 46, 47, 48, 12, 49, 27, 50, 30, 51],
 [20, 27, 52, 53, 54, 55, 56],
 [20, 57, 22, 58, 17, 59, 60, 48, 61, 62, 63, 64, '<unk>'],
 [46, 65, 17, 33, 66, 67, 68, 56, 28, 69, 70],
 [71, 72, 73, 11, 12, 74, 32, 75],
 [46, 76, 17, 77, 22, 63, 78, 79],
 [80, 81, 82, 83, 84, 85, 86, 35, 33, 87, 79, 88, 11, 28, 89],
 [20, 90, 91, 35, 20, 92, 91, 93],
 [20, 27, 11, 33, 94, 35, 95, 96, 97, 32, 28, 98, 99],
 [16, 100, 101, 11, 33, 102, 9, 10],
 [46, 65, 17, 103, 104, 35, 105, 106, 107, 32, 28, '<unk>'],
 [20, 21, 22, 108, 109, 28, 66, 110, 111],
 [20, 27, 112, 32, 12, 113, 56, 63, 114, 20, 115, 91, 116, 117],
 [77, 118, 119, 17, 120, 9, 121],
 [46, 122, 123, 100, 124, 125, 126, 82, 127],
 [20, 128, 129, 117, '<unk>', 130, 35, 112, 32, 33, 131],
 [46, 132, 133, 31, 1

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


In [ ]:
class Vanilla_RNN(nn.Module):
  def __init__(self,embed_size_en,vocab_size_en,hidden_size_en,hidden_size_de,vocab_size_de,embed_size_de):
    super(Vanilla_RNN,self).__init__()
    self.rnn_encode=nn.RNN(embed_size_en,hidden_size_en,batch_first=True,num_layers=1)
    self.embed_size_en=nn.Embedding(vocab_size_en,embed_size_en)
    self.rnn_decode=nn.RNN(embed_size_de,hidden_size_de,batch_first=True,num_layers=1)
    self.embed_size_de=nn.Embedding(vocab_size_de,embed_size_de)

    self.output=nn.Linear(hidden_size_de, vocab_size_de)
  def forward(self,x_en,x_de,h0=None):
    e_en=self.embed_size_en(x_en)
    out,hn=self.rnn_encode(e_en,h0)
    e_de=self.embed_size_de(x_de)
    out,hn_de=self.rnn_decode(e_de.unsqueeze(0), hn)
    logits=self.output(out.squeeze(0))
    return logits

